<a href="https://colab.research.google.com/github/madhurakatre55-collab/VioceX/blob/main/Music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install music21 torch numpy

In [3]:
import os
import numpy as np
from music21 import converter ,instrument,note,chord
drive_path = "/content/drive/MyDrive/Music_AI_Project/midi_dataset"
def parse_guiter_songs(data_folder):
  all_notes = []
  files = [f for f in os.listdir(data_folder) if f.endswith('.mid') or f.endswith('.midi')]
  for file in files:
      file_path = os.path.join(data_folder,file)
      try:
          midi = converter.parse(file_path)
          components = instrument.partitionByInstrument(midi)
          if components:
              notes_to_parse = components.parts[0].recurse()
          else:
              notes_to_parse = midi.data.recurse()
          for element in notes_to_parse:
              if isinstance(element,note.Note):
                  all_notes.append(str(element.pitch))
              elif isinstance(element,chord.Chord):
                  all_notes.append('.'.join(str(n) for n in element.normalOrder))
      except Exception as e:
          print(f"Error parsing {file}: {e}")
  return all_notes

guitar_sequence = parse_guiter_songs(drive_path)
print(guitar_sequence)

/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:1943: TranslateWarning: Unable to decode lyrics from <music21.midi.MidiEvent LYRIC, track=0, data=b'f\xe9'> as utf-8
  warnings.warn(


['D3', 'D3', 'A3', 'A3', 'D4', 'D4', 'A3', 'A3', 'E4', 'E4', 'D4', 'D4', 'A3', 'A3', 'D3', 'D3', 'F3', 'A3', 'E4', 'F3', 'A3', 'E4', 'A3', 'A3', 'D4', 'D4', 'F3', 'F3', 'D3', 'D4', 'D3', 'D4', 'A3', 'A3', 'D4', 'D4', 'A3', 'A3', 'E4', 'E4', 'D4', 'D4', 'A3', 'A3', 'D3', 'D3', 'F3', 'A3', 'E4', 'F3', 'A3', 'E4', 'A3', 'A3', 'D4', 'D4', 'F3', 'F3', 'D3', 'D4', 'A3', 'D3', 'D4', 'A3', 'A3', 'A3', 'D4', 'D4', 'A3', 'A3', 'E4', 'E4', 'D4', 'D4', 'A3', 'A3', 'D3', 'D3', 'F3', 'A3', 'E4', 'F3', 'A3', 'E4', 'A3', 'A3', 'D4', 'D4', 'F3', 'F3', 'D3', 'A3', 'D4', 'D3', 'A3', 'D4', 'A3', 'A3', 'D4', 'D4', 'A3', 'A3', 'E4', 'E4', 'D4', 'D4', 'A3', 'A3', 'D3', 'D3', 'F3', 'A3', 'E4', 'F3', 'A3', 'E4', 'A3', 'A3', 'D4', 'D4', 'E3', 'E3', 'G3', 'G3', 'D4', 'D4', 'C3', 'C3', 'E3', 'E3', 'D4', 'C3', 'E3', 'D4', 'C3', 'E3', 'G3', 'G3', 'E4', 'E4', 'D4', 'D4', 'G3', 'G3', 'E3', 'E3', 'C3', 'C3', 'E3', 'E3', 'D4', 'C3', 'E3', 'D4', 'C3', 'E3', 'G3', 'G3', 'E4', 'E4', 'D4', 'D4', 'F3', 'F3', 'C3', 'C3', 'D3

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

unique_pitcher = sorted(list(set(guitar_sequence)))
vocab_size = len(unique_pitcher)
note_to_int = dict((note,number) for number,note in enumerate(unique_pitcher))

sequence_length = 64
x_data = []
y_data = []

for i in range(0,len(guitar_sequence)-sequence_length):
  seq_in = guitar_sequence[i:i+sequence_length]
  seq_out = guitar_sequence[i+sequence_length]
  x_data.append([note_to_int[char] for char in seq_in])
  y_data.append(note_to_int[seq_out])

class GuitarDataset(Dataset):
  def __init__(self,x_data,y_data):
    self.x = torch.tensor(x_data, dtype=torch.long)
    self.y = torch.tensor(y_data, dtype=torch.long)
  def __len__(self):
    return len(self.x)
  def __getitem__(self,idx):
    return self.x[idx], self.y[idx]
dataset = GuitarDataset(x_data,y_data)
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)
print({len(x_data)})
print({vocab_size})

{5426}
{103}


In [6]:
import torch
import torch.nn as nn
#Rnn defined /Made
class LSTMModel(nn.Module):
  def __init__(self,vocab_size,embedding_dim,hidden_dim):
    super(LSTMModel,self).__init__()
    self.hidden_dim = hidden_dim # Corrected typo from hidden_din to hidden_dim
    self.embedding = nn.Embedding(vocab_size,embedding_dim)
    self.lstm = nn.LSTM(embedding_dim,hidden_dim,batch_first=True)
    self.fc = nn.Linear(hidden_dim,vocab_size)
  def init_hidden(self, batch_size, device):
    return (torch.zeros(1,batch_size,self.hidden_dim).to(device),
            torch.zeros(1,batch_size,self.hidden_dim).to(device))
  def forward(self, x, state=None, return_state=False):
    x = self.embedding(x)
    if state is None:
      state = self.init_hidden(x.size(0),x.device)
    out, state = self.lstm(x,state)
    out = self.fc(out[:,-1,:])
    return out if not return_state else (out,state)

vocab_size = 103
embedding_dim = 128
hidden_size = 256
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(vocab_size,embedding_dim,hidden_size).to(device)
print(model)

x,y = next(iter(dataloader))
x = x.to(device)
y = y.to(device)
pred = model(x)

print("Input shape",x.shape)
print("Prediction shape",pred.shape)
#input parameter initializes or models initialises
sampled_indices = torch.multinomial(torch.softmax(pred[0], dim=-1), num_samples=1)
sampled_indices = sampled_indices.squeeze(-1).cpu().numpy()
sampled_indices

idx2char = {v: k for k, v in note_to_int.items()}

print("Input:\n",repr("".join([idx2char[i.item()] for i in x[0].cpu()])))
print("Next Char Prediction:\n", repr(idx2char[sampled_indices.item()]))

LSTMModel(
  (embedding): Embedding(103, 128)
  (lstm): LSTM(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=103, bias=True)
)
Input shape torch.Size([32, 64])
Prediction shape torch.Size([32, 103])
Input:
 'A2D3B2B2A2C3C3D3A2B2E3E3C3D4D4E3G2G2D4G2G3G3D4D4G3G3G3D4C3G3C3G3G3G2D4G3E3E3C3D4D4E3G2G2D47.10.25.9.05.9.02.5.92.5.92.5.97.10.25.9.05.9.02.5.92.5.92.5.9C47.10.25.9.05.9.02.5.92.5.92.5.9'
Next Char Prediction:
 '9.0.4'


In [42]:
#for Loss calculations
import torch
import torch.nn as nn
cross_entropy = nn.CrossEntropyLoss()
def compute_loss(labels, logits):
  # labels: (batch_size, seq_len)
  # logits: (batch_size, seq_len, vocab_size)
  # loss: scalar
  batched_labels = labels.view(-1)
  batched_logits = logits.view(-1, logits.shape[-1])
  loss = cross_entropy(batched_logits, batched_labels)
  return loss
y.shape
pred.shape
compute_loss(y,pred)
example_loss = compute_loss(y,pred)
print(example_loss)

#Model Hyperparameters Parameters
params = dict(
    num_trainig_iterations = 30,
    batch_size = 8,
    seq_length = 100,
    learning_rate = 0.001,
    embedding_dim = 768,
    hidden_dim = 1024
)
checkpoint_dir = "/content/drive/MyDrive/Music_AI_Project/checkpoints"
os.makedirs(checkpoint_dir,exist_ok=True)

tensor(4.6442, device='cuda:0', grad_fn=<NllLossBackward0>)


In [43]:
#Comet experiment to track trainig
def create_experiment():
  if 'experiment' in locals():
    experiment.end()
  experiment = comet_ml.Experiment(
      api_key="",
      project_name=""
  )
  for param, value in params.items():
    experiment.log_parameter(param,value)
  experiment.flush()
  return experiment

In [44]:
import torch.optim as optim
from tqdm.auto import tqdm

model.to(device)

optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

def train_step(x, y):
  model.train()
  optimizer.zero_grad()
  y_hat = model(x)
  loss = compute_loss(y, y_hat)
  loss.backward()
  optimizer.step()
  return loss.item()

history = []
num_training_iterations = params['num_trainig_iterations']

print(f"Starting training for {num_training_iterations} epochs...")
for epoch in tqdm(range(num_training_iterations), desc="Training epochs"):
    epoch_loss = 0
    for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device);
        loss = train_step(x_batch, y_batch)
        epoch_loss += loss

    avg_epoch_loss = epoch_loss / len(dataloader)
    history.append(avg_epoch_loss)
    print(f"Epoch {epoch+1}/{num_training_iterations}, Average Loss: {avg_epoch_loss:.4f}")

print("Training complete.")

Starting training for 30 epochs...


Training epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/30, Average Loss: 0.0182
Epoch 2/30, Average Loss: 0.0271
Epoch 3/30, Average Loss: 0.0231
Epoch 4/30, Average Loss: 0.0193
Epoch 5/30, Average Loss: 0.0161
Epoch 6/30, Average Loss: 0.0176
Epoch 7/30, Average Loss: 0.0211
Epoch 8/30, Average Loss: 0.0146
Epoch 9/30, Average Loss: 0.0139
Epoch 10/30, Average Loss: 0.0105
Epoch 11/30, Average Loss: 0.0149
Epoch 12/30, Average Loss: 0.0113
Epoch 13/30, Average Loss: 0.0126
Epoch 14/30, Average Loss: 0.0115
Epoch 15/30, Average Loss: 0.0109
Epoch 16/30, Average Loss: 0.0097
Epoch 17/30, Average Loss: 0.0085
Epoch 18/30, Average Loss: 0.0082
Epoch 19/30, Average Loss: 0.0095
Epoch 20/30, Average Loss: 0.0104
Epoch 21/30, Average Loss: 0.0961
Epoch 22/30, Average Loss: 0.0773
Epoch 23/30, Average Loss: 0.0364
Epoch 24/30, Average Loss: 0.0255
Epoch 25/30, Average Loss: 0.0155
Epoch 26/30, Average Loss: 0.0164
Epoch 27/30, Average Loss: 0.0145
Epoch 28/30, Average Loss: 0.0113
Epoch 29/30, Average Loss: 0.0096
Epoch 30/30, Average Lo

In [47]:
def generate_text(model, start_string, generation_length=1000):
  model.eval() # Set the model to evaluation mode

  # Convert start_string to a list containing its integer index
  input_indices_list = [note_to_int[start_string]]

  # Initialize text_generated with the start_string
  text_generated = [start_string]

  # Use the global sequence_length defined during training (from cell O8IOu_oobZG3)
  # This ensures the input to the model for generation has the same length as during training.
  global sequence_length # Assuming sequence_length is available in the global scope
  global idx2char # Assuming idx2char is available in the global scope

  # To initialize the state, we first need an input tensor from the start_string.
  # For the first pass, we feed the entire `start_string` to prime the state.
  # We need to ensure `initial_input_tensor` has at least `sequence_length` elements for consistency,
  # but `input_indices_list[-sequence_length:]` handles shorter lists by returning the list itself.
  initial_input_for_state = input_indices_list[-sequence_length:] if len(input_indices_list) >= sequence_length else input_indices_list
  initial_input_tensor = torch.tensor(initial_input_for_state, dtype=torch.long).unsqueeze(0).to(device)

  # Initialize the hidden state for a batch size of 1.
  state = model.init_hidden(initial_input_tensor.size(0), initial_input_tensor.device)

  # Clear tqdm instances to prevent display issues in Colab
  tqdm._instances.clear()

  # Generation loop
  with torch.no_grad(): # Disable gradient calculation for inference
    for _ in tqdm(range(generation_length), desc="Generating text"):
      # Prepare the input tensor for the current step.
      # It should be the last `sequence_length` tokens from the sequence generated so far.
      current_input_sequence = input_indices_list[-sequence_length:]

      # Convert to tensor for model input
      current_input_tensor = torch.tensor(current_input_sequence, dtype=torch.long).unsqueeze(0).to(device)

      # Get model output and update state, explicitly asking for the state to be returned
      output, state = model(current_input_tensor, state, return_state=True)

      # Sample the next character
      output_dist = torch.distributions.Categorical(logits=output) # Use logits directly for Categorical
      sampled_index = output_dist.sample() # This samples a tensor like `tensor([51], device='cuda:0')`

      predicted_note_int = sampled_index.item() # Get the integer value

      # Append the new integer index to our sequence list
      input_indices_list.append(predicted_note_int)

      # Convert the integer index back to a note string and append to text_generated
      text_generated.append(idx2char[predicted_note_int])

  # Join the generated characters starting from the original `start_string` length
  # to return only the newly generated part appended to the start_string.
  return ''.join(text_generated)

In [48]:
from music21 import stream, instrument, note, chord
from IPython.display import Audio, display as ipythondisplay

def parse_generated_string(generated_str, unique_pitches):
    """
    Parses a concatenated string of notes/chords back into a list of individual note/chord strings.
    This function relies on a predefined list of unique_pitches (notes/chords)
    to correctly segment the generated string.
    """
    parsed_elements = []
    current_index = 0
    sorted_unique_pitches = sorted(list(unique_pitches), key=len, reverse=True)

    while current_index < len(generated_str):
        matched = False
        for pitch_str in sorted_unique_pitches:
            if generated_str.startswith(pitch_str, current_index):
                parsed_elements.append(pitch_str)
                current_index += len(pitch_str)
                matched = True
                break
        if not matched:
            print(f"Warning: Could not parse part of the generated string: '{generated_str[current_index:]}' at index {current_index}. Skipping character.")
            current_index += 1
    return parsed_elements

start_string_for_generation = 'C4'
if start_string_for_generation not in note_to_int:
    print(f"Start string '{start_string_for_generation}' not in vocabulary. Using '{unique_pitcher[0]}' instead.")
    start_string_for_generation = unique_pitcher[0]

generation_length = 150
print(f"Generating a musical sequence of {generation_length} elements starting with '{start_string_for_generation}'...")
generated_song_str = generate_text(model, start_string_for_generation, generation_length=generation_length);

generated_notes_list = parse_generated_string(generated_song_str, unique_pitcher)

output_stream = stream.Stream()

output_stream.append(instrument.AcousticGuitar())

for element_str in generated_notes_list:
    try:
        if '.' in element_str:
            pitch_classes = [int(pc_str) for pc_str in element_str.split('.')]
            chord_notes = [note.Note(pitch_class=pc, octave=4) for pc in pitch_classes]
            output_stream.append(chord.Chord(chord_notes))
        else:
            output_stream.append(note.Note(element_str))
    except Exception as e:
        output_stream.append(note.Rest())

print("\nPlaying generated music:")
ipythondisplay(output_stream.show('midi'))

midi_output_path = '/content/generated_song.midi'
output_stream.write('midi', fp=midi_output_path)
print(f"MIDI file saved to {midi_output_path}")

Generating a musical sequence of 150 elements starting with 'C4'...


Generating text:   0%|          | 0/150 [00:00<?, ?it/s]


Playing generated music:


None

MIDI file saved to /content/generated_song.midi


In [41]:
model_save_path = os.path.join(checkpoint_dir, 'music_lstm_model.pth')
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to /content/drive/MyDrive/Music_AI_Project/checkpoints/music_lstm_model.pth
